# 02 · Limpieza y preparación de datos

**Proyecto:** Predicción de riesgo de burnout organizacional · **Autor:** Oscar Paúl Naranjo Castro
**Maestría en Inteligencia Artificial, UEES** · Todas las rutas son relativas a la raíz del repositorio.

Se aplican las decisiones del EDA con las funciones reutilizables de `src/data_processing.py`.

In [1]:
%matplotlib inline
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

In [2]:
from src.data_processing import (FEATURES, TARGET, build_preprocessor, clean_data, load_raw_data,
                                 save_processed, split_data)
raw = load_raw_data()
df = clean_data(raw)
print('Filas originales:', len(raw), '| filas con objetivo:', len(df))
df.head()

14:04:46 | INFO | src.data_processing | Datos cargados: 22750 filas, 9 columnas


14:04:46 | INFO | src.data_processing | Filas con objetivo observado: 21626 (se descartaron 1124)


Filas originales: 22750 | filas con objetivo: 21626


,Designation,Resource Allocation,Mental Fatigue Score,Company Type,WFH Setup Available,Gender,Burn Rate
0,2.0,3.0,3.8,Service,No,Female,0.16
1,1.0,2.0,5.0,Service,Yes,Male,0.36
2,2.0,NaN,5.8,Product,Yes,Female,0.49
3,1.0,1.0,2.6,Service,Yes,Male,0.20
4,3.0,7.0,6.9,Service,No,Female,0.52


## Partición estratificada 80/20
Se estratifica por la alerta (Burn Rate ≥ 0.6) para conservar la proporción de riesgo alto.

In [3]:
X_train, X_test, y_train, y_test = split_data(df, 0.2, 42)
pd.DataFrame({'n': [len(y_train), len(y_test)],
              'pct_riesgo_alto': [(y_train >= .6).mean(), (y_test >= .6).mean()],
              'pct_fatiga_faltante': [X_train['Mental Fatigue Score'].isna().mean(), X_test['Mental Fatigue Score'].isna().mean()]},
             index=['train', 'test']).round(4)

,n,pct_riesgo_alto,pct_fatiga_faltante
train,17300,0.2383,0.0887
test,4326,0.2383,0.0950


## Preprocesador (ajustado solo con train)
* Numéricas: mediana + indicador de dato faltante.
* Categóricas: one-hot con categoría de referencia.
* *Gender* queda fuera del modelo.

In [4]:
prep = build_preprocessor().fit(X_train)
print('Medianas aprendidas en train:', dict(zip(prep.named_transformers_['num'].feature_names_in_,
                                              prep.named_transformers_['num'].statistics_)))
Xt = prep.transform(X_train)
print('Columnas resultantes:', list(Xt.columns))
Xt.head()

Medianas aprendidas en train: {'Designation': np.float64(2.0), 'Resource Allocation': np.float64(4.0), 'Mental Fatigue Score': np.float64(5.9)}
Columnas resultantes: ['Designation', 'Resource Allocation', 'Mental Fatigue Score', 'missingindicator_Resource Allocation', 'missingindicator_Mental Fatigue Score', 'Company Type_Service', 'WFH Setup Available_Yes']


,Designation,Resource Allocation,Mental Fatigue Score,missingindicator_Resource Allocation,missingindicator_Mental Fatigue Score,Company Type_Service,WFH Setup Available_Yes
6480,3.0,5.0,7.0,0.0,0.0,0.0,0.0
14537,2.0,4.0,5.5,0.0,0.0,1.0,1.0
7171,1.0,4.0,5.0,1.0,0.0,1.0,0.0
14628,2.0,4.0,7.0,0.0,0.0,0.0,1.0
12430,2.0,4.0,6.3,0.0,0.0,1.0,1.0


In [5]:
assert not Xt.isna().any().any(), 'Quedaron faltantes después de transformar'
assert not any('Gender' in c for c in Xt.columns)
print('Sin faltantes ni variable de género tras el preprocesamiento.')

Sin faltantes ni variable de género tras el preprocesamiento.


## Verificación de ausencia de fuga de datos
Las medianas se calculan solo con train; aplicarlas al test no usa información del test.

In [6]:
med_train = X_train[['Resource Allocation', 'Mental Fatigue Score']].median()
med_full = df[['Resource Allocation', 'Mental Fatigue Score']].median()
pd.DataFrame({'mediana_train': med_train, 'mediana_todo': med_full})

,mediana_train,mediana_todo
Resource Allocation,4.0,4.0
Mental Fatigue Score,5.9,5.9


In [7]:
out = save_processed(df)
print('Guardado en', out.relative_to(ROOT))

14:04:46 | INFO | src.data_processing | Dataset procesado guardado en /home/claude/opnaranjo/proyecto_integrador_ia/data/processed/burnout_clean.csv


Guardado en data/processed/burnout_clean.csv


## Resumen
| Decisión | Justificación |
|---|---|
| Eliminar 1 124 filas sin objetivo | No se fabrican etiquetas |
| Descartar ID y fecha | Sin valor predictivo (fecha solo 2008, ρ = 0.001) |
| Mediana + indicador para fatiga y carga | Faltantes compatibles con MCAR; el indicador permite tratar distinto los casos imputados |
| One-hot de tipo de empresa y teletrabajo | Variables nominales binarias |
| Excluir género | Aporta apenas el 0.2 % de la ganancia y no mejora el error; se usa solo para auditar |
| Todo dentro de un `Pipeline` | Evita fuga de información en validación cruzada |